# NFL Spread Market Efficiency Study
### Do publicly available metrics contain information not already priced into betting lines?

**Author:** Padil · Stanford Data Science
**Data:** 2010–2024 NFL regular season (nfl_data_py + SBR historical lines)
**Method:** Logistic regression with walk-forward validation

---

## Motivation

The NFL spread betting market is one of the most liquid and information-rich prediction markets in sports.
Professional syndicates, quant shops, and sharp bettors spend enormous resources modeling games.
The question this project asks is straightforward:

> **Can a model built on freely available metrics — rolling scoring differentials, Expected Points Added (EPA),
> weather, and opening line movement — beat a 52.38% win rate consistently enough to be profitable after -110 juice?**

This is framed deliberately as a **market efficiency study**, not a prediction tool.
If the answer is no, that's the interesting finding — it means the market has already priced in everything
a publicly available dataset can tell you.

### What breakeven looks like

Standard spread bets pay -110: risk \$110 to win \$100. The breakeven win rate is:

$$\text{Breakeven} = \frac{110}{110 + 100} = 52.38\%$$

A model that picks winners 53% of the time *sounds* good. But 53% × 100 - 47% × 110 = **-$1.70 per bet.**
You need sustained accuracy above 52.38% just to break even — and that bar is very hard to clear with public data.


---
## 1. Setup & Imports


In [ ]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
import os
import re
import openpyxl
from datetime import date as date_cls
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Constants
SEASONS     = list(range(2010, 2025))
DOME_TEAMS  = ['NO', 'ATL', 'IND', 'DET', 'MIN', 'LV', 'DAL', 'ARI']

# Team abbreviation normalization: PBP data uses current abbreviations;
# older schedule data still uses OAK, SD, STL for relocated franchises.
# Without this fix, EPA merges silently fail for ~184 games.
TEAM_ABBREV_MAP = {
    'JAC': 'JAX', 'LVR': 'LV', 'STL': 'LA', 'SD': 'LAC',
    'OAK': 'LV', 'ARZ': 'ARI', 'BLT': 'BAL', 'CLV': 'CLE', 'HST': 'HOU',
}

print("Libraries loaded.")


---
## 2. SBR Opening Line Data

Opening lines from [SBR Historical Odds](https://www.sportsbookreview.com/betting-odds/nfl-football/) (2010–2021)
were manually compiled into Excel files. The parser below handles the two-row-per-game format,
sign convention differences, and copy-paste artifacts across 12 seasons.

**Key sign convention:** positive spread_line = home team is favored (gives points).
This matches nfl_data_py's convention. We validate by checking correlation between the SBR close
and nfl_data_py's spread_line — they should agree at r ≈ 0.97.

The variable we ultimately test (`line_movement`) is how much the spread shifted from open to close.
Large moves signal sharp ("smart money") action. The hypothesis: if sharp bettors know something,
does it show up as a predictive signal for the final outcome?


In [ ]:
SBR_DIR = r'C:\Users\padil\OneDrive\Documents\nfl-model\sbr_data'

SBR_TEAM_MAP = {
    'Arizona': 'ARI', 'Atlanta': 'ATL', 'Baltimore': 'BAL', 'Buffalo': 'BUF',
    'Carolina': 'CAR', 'Chicago': 'CHI', 'Cincinnati': 'CIN', 'Cleveland': 'CLE',
    'Dallas': 'DAL', 'Denver': 'DEN', 'Detroit': 'DET', 'GreenBay': 'GB',
    'Houston': 'HOU', 'Indianapolis': 'IND', 'Jacksonville': 'JAX',
    'KansasCity': 'KC', 'LAChargers': 'LAC', 'LARams': 'LA', 'LasVegas': 'LV',
    'Miami': 'MIA', 'Minnesota': 'MIN', 'NewEngland': 'NE', 'NewOrleans': 'NO',
    'NYGiants': 'NYG', 'NYJets': 'NYJ', 'Oakland': 'LV', 'Philadelphia': 'PHI',
    'Pittsburgh': 'PIT', 'SanDiego': 'LAC', 'SanFrancisco': 'SF',
    'Seattle': 'SEA', 'StLouis': 'LA', 'TampaBay': 'TB', 'Tennessee': 'TEN',
    'Washington': 'WAS', 'WashingtonFootball': 'WAS', 'Commanders': 'WAS',
    'St.Louis': 'LA', 'LosAngeles': 'LA', 'LVRaiders': 'LV', 'KCChiefs': 'KC',
    'Kansas': 'KC', 'Tampa': 'TB', 'BuffaloBills': 'BUF', 'NewYork': 'NYG',
    'Washingtom': 'WAS',
}

def _parse_spread_val(val):
    if val is None: return np.nan
    if str(val).strip().lower() in ('pk', 'ev', 'pick'): return 0.0
    try: return float(val)
    except: return np.nan

def _is_spread(val):
    if val is None: return False
    if str(val).strip().lower() in ('pk', 'ev', 'pick'): return True
    try: return abs(float(val)) < 30
    except: return False

def _season_year_from_filename(path):
    name = os.path.basename(path)
    m = re.search(r'(\d{2})-\d{2}', name)
    if m: return 2000 + int(m.group(1))
    m = re.search(r'(\d{4})', name)
    if m: return int(m.group(1))
    raise ValueError(f"Cannot extract season year from: {name}")

def _mmdd_to_date(mmdd_val, season_year):
    mmdd = int(mmdd_val)
    month, day = mmdd // 100, mmdd % 100
    year = season_year if month >= 3 else season_year + 1
    return str(date_cls(year, month, day))

def parse_sbr_file(path):
    season_year = _season_year_from_filename(path)
    wb = openpyxl.load_workbook(path)
    ws = wb.active
    data = [r for r in ws.iter_rows(values_only=True) if r[2] not in (None, 'VH')]
    games, i = [], 0
    while i < len(data) - 1:
        r1, r2 = data[i], data[i+1]
        if r1[2] == 'N' or r2[2] == 'N': i += 1; continue
        if   r1[2] == 'V' and r2[2] == 'H': v_row, h_row = r1, r2
        elif r1[2] == 'H' and r2[2] == 'V': h_row, v_row = r1, r2
        else: i += 1; continue
        date_val = v_row[0] or h_row[0]
        if date_val is None: i += 2; continue
        v_open, v_close = v_row[9], v_row[10]
        h_open, h_close = h_row[9], h_row[10]
        v_is_spread, h_is_spread = _is_spread(v_open), _is_spread(h_open)
        if   h_is_spread and not v_is_spread:
            spread_open, spread_close = _parse_spread_val(h_open), _parse_spread_val(h_close)
        elif v_is_spread and not h_is_spread:
            spread_open  = -_parse_spread_val(v_open)
            spread_close = -_parse_spread_val(v_close)
        else: i += 2; continue
        if spread_close is not None and abs(spread_close or 0) >= 30:
            spread_close = spread_open
        away_team = SBR_TEAM_MAP.get(str(v_row[3]).strip(), str(v_row[3]).strip())
        home_team = SBR_TEAM_MAP.get(str(h_row[3]).strip(), str(h_row[3]).strip())
        try: gameday = _mmdd_to_date(date_val, season_year)
        except: i += 2; continue
        games.append({'gameday': gameday, 'away_team': away_team, 'home_team': home_team,
                      'spread_open': spread_open, 'spread_close_sbr': spread_close})
        i += 2
    return pd.DataFrame(games)

def load_all_sbr(sbr_dir):
    frames = []
    for fname in sorted(os.listdir(sbr_dir)):
        if fname.endswith('.xlsx') or fname.endswith('.xls'):
            path = os.path.join(sbr_dir, fname)
            try:
                df = parse_sbr_file(path)
                frames.append(df)
                print(f"  Loaded {fname}: {len(df)} games")
            except Exception as e:
                print(f"  SKIP {fname}: {e}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("SBR parser defined.")


---
## 3. Data Loading

We load two datasets from `nfl_data_py`:
- **Schedules** (2010–2024): game-level results, spreads, weather, rest days
- **Play-by-play (PBP)**: every play with EPA values, used to compute team efficiency metrics

The target variable is `home_covered = 1` when the home team beats the spread.

> **Critical fix:** nfl_data_py uses current abbreviations (LV, LAC, LA) in PBP data,
> but historical schedule data still uses OAK, SD, STL for relocated franchises.
> Without normalizing abbreviations before the merge, ~184 games silently lose their EPA values.


In [ ]:
print("Loading schedules 2010-2024...")
schedules = nfl.import_schedules(SEASONS)
schedules = schedules[schedules['game_type'] == 'REG'].copy()

# Target variable: did home team cover the spread?
schedules['home_covered'] = (schedules['result'] > schedules['spread_line']).astype(int)

# Normalize abbreviations BEFORE merging EPA (prevents 184 silent failures)
schedules['home_team'] = schedules['home_team'].replace(TEAM_ABBREV_MAP)
schedules['away_team'] = schedules['away_team'].replace(TEAM_ABBREV_MAP)

# Weather features
schedules['temp_known'] = schedules['temp'].notna()
schedules['temp']  = schedules['temp'].fillna(70)   # dome/missing → neutral temp
schedules['wind']  = schedules['wind'].fillna(0)

# Derived features
schedules['rest_diff']          = schedules['home_rest'] - schedules['away_rest']
schedules['home_is_favorite']   = (schedules['spread_line'] < 0).astype(int)
schedules['large_favorite']     = (schedules['spread_line'] < -7).astype(int)
schedules['dome_team_outdoors'] = (
    schedules['home_team'].isin(DOME_TEAMS) &
    schedules['temp_known'] &
    (schedules['temp'] < 40)
).astype(int)

schedules = schedules.sort_values('gameday').reset_index(drop=True)
print(f"Schedules loaded: {len(schedules):,} regular season games")
print(f"Home cover rate:  {schedules['home_covered'].mean():.1%}  (historical baseline)")


---
## 4. Feature Engineering

### 4a. Rolling Scoring Differential

A team's recent offensive and defensive form, measured as a rolling 4-game average of
**(points scored − points allowed)**.

**Why rolling per season?** Crossing a season boundary makes no sense — a team's Week 1 performance
should not factor into its Week 17 rolling average from the prior season.
We use `groupby(['team', 'season'])` to reset the window at each new season.

**Why shift(1)?** We shift by one game before computing the mean to avoid lookahead bias
(the current game's result can't be known before it's played).


In [ ]:
# Reshape to team-level (one row per team per game)
home_games = schedules[['game_id','gameday','season','week','home_team','away_team','home_score','away_score']].copy()
home_games.columns = ['game_id','gameday','season','week','team','opponent','points_scored','points_allowed']
away_games = schedules[['game_id','gameday','season','week','away_team','home_team','away_score','home_score']].copy()
away_games.columns = ['game_id','gameday','season','week','team','opponent','points_scored','points_allowed']

game_log = pd.concat([home_games, away_games]).sort_values('gameday').reset_index(drop=True)

# Rolling 4-game avg within season (no lookahead)
game_log['rolling_scored']  = game_log.groupby(['team','season'])['points_scored'].transform(
    lambda x: x.shift(1).rolling(4, min_periods=1).mean())
game_log['rolling_allowed'] = game_log.groupby(['team','season'])['points_allowed'].transform(
    lambda x: x.shift(1).rolling(4, min_periods=1).mean())
game_log['rolling_diff'] = game_log['rolling_scored'] - game_log['rolling_allowed']

team_stats = game_log[['game_id','team','rolling_diff']].copy()

schedules = schedules.merge(
    team_stats.rename(columns={'team':'home_team','rolling_diff':'home_rolling_diff'}),
    on=['game_id','home_team'], how='left')
schedules = schedules.merge(
    team_stats.rename(columns={'team':'away_team','rolling_diff':'away_rolling_diff'}),
    on=['game_id','away_team'], how='left')
schedules['diff_rolling_diff'] = schedules['home_rolling_diff'] - schedules['away_rolling_diff']

print("Rolling scoring differential computed.")


### 4b. Expected Points Added (EPA)

EPA measures how much each play increased or decreased a team's expected points based on
down, distance, and field position. It's a better efficiency metric than raw scoring because
it controls for game situation.

We compute:
- **Offensive EPA** per play: how efficiently the team's offense generated expected points
- **Defensive EPA** per play: how much EPA the team's defense *allowed* (lower = better defense)

Both are computed as rolling 4-game averages with the same season-reset and shift-by-1 logic.


In [ ]:
print("Loading play-by-play data (this takes 3-5 minutes)...")
pbp = nfl.import_pbp_data(SEASONS)
pbp = pbp[pbp['play_type'].isin(['pass','run']) & pbp['epa'].notna()].copy()
pbp['posteam'] = pbp['posteam'].replace(TEAM_ABBREV_MAP)
pbp['defteam'] = pbp['defteam'].replace(TEAM_ABBREV_MAP)

offense_epa = pbp.groupby(['game_id','posteam'])['epa'].mean().reset_index()
offense_epa.columns = ['game_id','team','off_epa_per_play']
defense_epa = pbp.groupby(['game_id','defteam'])['epa'].mean().reset_index()
defense_epa.columns = ['game_id','team','def_epa_per_play']

team_epa = offense_epa.merge(defense_epa, on=['game_id','team'])
team_epa = team_epa.merge(
    schedules[['game_id','gameday','season']].drop_duplicates(), on='game_id'
).sort_values('gameday').reset_index(drop=True)

team_epa['rolling_off_epa'] = team_epa.groupby(['team','season'])['off_epa_per_play'].transform(
    lambda x: x.shift(1).rolling(4, min_periods=1).mean())
team_epa['rolling_def_epa'] = team_epa.groupby(['team','season'])['def_epa_per_play'].transform(
    lambda x: x.shift(1).rolling(4, min_periods=1).mean())

schedules = schedules.merge(
    team_epa[['game_id','team','rolling_off_epa','rolling_def_epa']].rename(columns={
        'team':'home_team','rolling_off_epa':'home_rolling_off_epa','rolling_def_epa':'home_rolling_def_epa'}),
    on=['game_id','home_team'], how='left')
schedules = schedules.merge(
    team_epa[['game_id','team','rolling_off_epa','rolling_def_epa']].rename(columns={
        'team':'away_team','rolling_off_epa':'away_rolling_off_epa','rolling_def_epa':'away_rolling_def_epa'}),
    on=['game_id','away_team'], how='left')

schedules['diff_off_epa'] = schedules['home_rolling_off_epa'] - schedules['away_rolling_off_epa']
schedules['diff_def_epa'] = schedules['home_rolling_def_epa'] - schedules['away_rolling_def_epa']
print("EPA features computed.")


### 4c. QB-Adjusted EPA (Investigated but Excluded)

**Hypothesis:** Team-level EPA is noisy because backup QB games drag down the rolling average.
If we restrict EPA calculations to plays by the *season starter*, we might isolate a cleaner signal
— and additionally flag when a starter is injured/absent recently.

**Method:**
1. Game-day starter = QB with most pass attempts in each game
2. Season starter = QB who started the most games that season
3. Starter EPA = EPA restricted to the season starter's throws
4. `starter_played` = fraction of last 4 games the season starter actually played

**Result:** After all the engineering, both features showed no correlation with home_covered
(p = 0.977 and p = 0.274). Worse, this approach drops **37% of games** due to incomplete PBP
passer data in early seasons. The sample size cost wasn't worth it — these features were excluded from the final model.

This is a useful negative result: it shows the market already adjusts for known QB injuries.


In [ ]:
pass_plays = pbp[(pbp['play_type'] == 'pass') & (pbp['passer_player_id'].notna())].copy()

# Step 1: game-day starter (QB with most attempts per game)
qb_attempts = pass_plays.groupby(['game_id','posteam','passer_player_id']).size().reset_index(name='attempts')
qb_attempts = qb_attempts.sort_values('attempts', ascending=False)
game_starters = (qb_attempts.groupby(['game_id','posteam']).first().reset_index()
    [['game_id','posteam','passer_player_id']]
    .rename(columns={'posteam':'team','passer_player_id':'game_starter_id'}))

# Step 2: season starter (most common game-day starter per team per season)
game_starters = game_starters.merge(pass_plays[['game_id','season']].drop_duplicates(), on='game_id')
season_starter_counts = (game_starters.groupby(['season','team','game_starter_id'])
    .size().reset_index(name='games_started').sort_values('games_started', ascending=False))
season_starters = (season_starter_counts.groupby(['season','team']).first().reset_index()
    [['season','team','game_starter_id']].rename(columns={'game_starter_id':'season_starter_id'}))

game_starters = game_starters.merge(season_starters, on=['season','team'])
game_starters['starter_played'] = (game_starters['game_starter_id'] == game_starters['season_starter_id']).astype(int)

# Step 3: EPA restricted to season starter throws
starter_plays = pass_plays.merge(season_starters.rename(columns={'team':'posteam'}), on=['season','posteam'])
starter_plays = starter_plays[starter_plays['passer_player_id'] == starter_plays['season_starter_id']]
starter_epa = (starter_plays.groupby(['game_id','posteam'])['epa'].mean().reset_index()
    .rename(columns={'posteam':'team','epa':'starter_off_epa_per_play'}))
starter_epa = starter_epa.merge(game_starters[['game_id','team','starter_played']], on=['game_id','team'])
starter_epa = starter_epa.merge(
    schedules[['game_id','gameday','season']].drop_duplicates(), on='game_id'
).sort_values('gameday').reset_index(drop=True)

# Step 4: rolling (per season, no lookahead)
starter_epa['rolling_starter_off_epa'] = (starter_epa.groupby(['team','season'])['starter_off_epa_per_play']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()))
starter_epa['rolling_starter_played']  = (starter_epa.groupby(['team','season'])['starter_played']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()))

# Step 5: merge into schedules
for side in [('home_team','home'), ('away_team','away')]:
    col, prefix = side
    schedules = schedules.merge(
        starter_epa[['game_id','team','rolling_starter_off_epa','rolling_starter_played']].rename(columns={
            'team': col,
            'rolling_starter_off_epa': f'{prefix}_rolling_starter_off_epa',
            'rolling_starter_played':  f'{prefix}_rolling_starter_played',
        }), on=['game_id', col], how='left')

schedules['diff_starter_off_epa'] = schedules['home_rolling_starter_off_epa'] - schedules['away_rolling_starter_off_epa']
schedules['diff_starter_played']  = schedules['home_rolling_starter_played']  - schedules['away_rolling_starter_played']

# Assess cost: how many games have valid QB data?
n_with_qb = schedules[['diff_starter_off_epa','diff_starter_played']].dropna().shape[0]
n_total = len(schedules)
print(f"Games with QB features: {n_with_qb:,} / {n_total:,} ({n_with_qb/n_total:.1%})")
print(f"Games lost to missing QB data: {n_total - n_with_qb:,} ({(n_total-n_with_qb)/n_total:.1%})")

# Correlation with home_covered
for col in ['diff_starter_off_epa', 'diff_starter_played']:
    sub = schedules[['home_covered', col]].dropna()
    r, p = stats.pearsonr(sub[col], sub['home_covered'])
    print(f"  {col}: r={r:+.4f}, p={p:.3f}  ({'significant' if p < 0.05 else 'NOT significant'})")

print("\nConclusion: QB features dropped — 37% sample loss with no predictive signal.")


### 4d. SBR Line Movement

Opening lines from SBR (2010–2021). We validate by checking that `spread_close_sbr` correlates
tightly with `nfl_data_py`'s `spread_line` (r ≈ 0.97 confirms sign convention is correct).

`line_movement = spread_close − spread_open`. A large positive value means the home team
became a bigger favorite from open to close — a signal of sharp money on the home side.

**Result:** r = −0.004, p = 0.858 — essentially zero correlation with home covering.
The betting market quickly incorporates new information; by game time there's nothing left to exploit.


In [ ]:
print("Loading SBR line data...")
sbr = load_all_sbr(SBR_DIR)
print(f"\nTotal SBR games loaded: {len(sbr):,}")

if not sbr.empty:
    schedules = schedules.merge(
        sbr[['gameday','away_team','home_team','spread_open','spread_close_sbr']],
        on=['gameday','away_team','home_team'], how='left')

    check = schedules[['spread_line','spread_close_sbr']].dropna()
    corr = check.corr().iloc[0,1]
    print(f"\nValidation: SBR close vs nfl_data_py spread_line: r = {corr:.3f}")
    if corr < 0:
        print("  Signs flipped — negating SBR spreads")
        schedules['spread_open']      = -schedules['spread_open']
        schedules['spread_close_sbr'] = -schedules['spread_close_sbr']

    schedules['line_movement'] = schedules['spread_close_sbr'] - schedules['spread_open']
    n_lines = schedules['line_movement'].notna().sum()
    print(f"Games with line movement data: {n_lines:,} / {len(schedules):,}")

    sub = schedules[['home_covered','line_movement']].dropna()
    r, p = stats.pearsonr(sub['line_movement'], sub['home_covered'])
    print(f"\nLine movement vs home_covered: r = {r:+.4f}, p = {p:.3f}")
    print("Conclusion: no signal — sharp money does not predict the final outcome relative to spread.")
else:
    schedules['line_movement'] = np.nan
    print("No SBR files found.")


---
## 5. Feature Signal Analysis

Before modeling, we test each feature's raw correlation with `home_covered` using Pearson r
and a two-tailed p-value. This tells us whether any individual feature contains
statistically significant information about outcomes.

**Bonferroni-corrected significance threshold** for 9 features: α/9 ≈ 0.0056.
At the conventional α = 0.05, we'd expect ~0.45 false positives by chance alone.

This analysis reveals the core finding of the project:
**only one feature** clears the conventional significance threshold.


In [ ]:
_base_features = [
    'diff_def_epa',        # defensive EPA differential (home − away)
    'diff_rolling_diff',   # rolling scoring differential (home − away)
    'large_favorite',      # home team favored by 7+ points
    'temp',                # game-time temperature
    'home_is_favorite',    # binary: home team is favored
    'diff_off_epa',        # offensive EPA differential (home − away)
    'rest_diff',           # rest days differential (home − away)
    'wind',                # wind speed at game time
    'dome_team_outdoors',  # dome team playing in cold (< 40°F) outdoor game
]

FEATURES = _base_features

model_data = schedules.dropna(subset=FEATURES + ['home_covered']).copy()

corr_rows = []
for f in FEATURES:
    r, p = stats.pearsonr(model_data[f].fillna(0), model_data['home_covered'])
    corr_rows.append({'Feature': f, 'r': round(r, 4), 'p-value': round(p, 4),
                      'Significant': '✓' if p < 0.05 else ''})
corr_df = pd.DataFrame(corr_rows).sort_values('p-value')

print(f"Dataset: {len(model_data):,} games (dropped {len(schedules)-len(model_data):,} with missing features)")
print(f"  ~241 from week 1 rolling NaN (no prior-season data)\n")
print(corr_df.to_string(index=False))
print(f"\nNote: only diff_def_epa clears p < 0.05.")
print(f"With Bonferroni correction (α/9 ≈ 0.006), even that is borderline.")


---
## 6. Walk-Forward Validation

### Methodology

Walk-forward validation mimics real-world deployment: we only train on data available
*before* the test season. For each season from 2014 onward:

1. **Train** on all prior seasons (e.g., for 2014: train on 2010–2013)
2. **Test** on that season's games using a model never seen that data
3. Record accuracy vs. the naive baseline (always predict the historical cover rate)

This is stricter than k-fold cross-validation, which can inadvertently let future data
leak into training. Walk-forward is the only defensible approach for time-series data.

**Model:** Logistic regression with strong regularization (C=0.01).
High regularization prevents the model from overfitting to noise features — with C=0.01,
statistically insignificant features are effectively zeroed out by the penalty.


In [ ]:
def walk_forward(model_data, features, start=2014, end=2025):
    season_results = []
    all_preds = []

    for test_season in range(start, end):
        train = model_data[model_data['season'] < test_season]
        test  = model_data[model_data['season'] == test_season].copy()
        if len(train) < 100 or len(test) == 0:
            continue

        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[features])
        X_test  = scaler.transform(test[features])

        clf = LogisticRegression(C=0.01, max_iter=1000)
        clf.fit(X_train, train['home_covered'])

        test['pred']      = clf.predict(X_test)
        test['prob_home'] = clf.predict_proba(X_test)[:, 1]

        season_results.append({
            'season':   test_season,
            'games':    len(test),
            'accuracy': accuracy_score(test['home_covered'], test['pred']),
            'baseline': test['home_covered'].mean(),
        })
        all_preds.append(test)

    return pd.DataFrame(season_results), pd.concat(all_preds)

res, predictions = walk_forward(model_data, FEATURES)

# Display results
print(f"  {'Season':>6}  {'Games':>5}  {'Accuracy':>9}  {'Baseline':>9}  {'vs Baseline':>12}")
print(f"  {'─'*6}  {'─'*5}  {'─'*9}  {'─'*9}  {'─'*12}")
for _, row in res.iterrows():
    vs = row['accuracy'] - row['baseline']
    marker = ' ▲' if vs > 0 else ' ▼'
    print(f"  {int(row['season']):>6}  {int(row['games']):>5}  "
          f"{row['accuracy']:>9.1%}  {row['baseline']:>9.1%}  {vs:>+11.1%}{marker}")
print(f"  {'─'*6}  {'─'*5}  {'─'*9}  {'─'*9}  {'─'*12}")
print(f"  {'MEAN':>6}  {'':>5}  {res['accuracy'].mean():>9.1%}  "
      f"{res['baseline'].mean():>9.1%}  "
      f"{(res['accuracy'] - res['baseline']).mean():>+11.1%}")
print(f"  Seasons above baseline: {(res['accuracy'] > res['baseline']).sum()} / {len(res)}")


---
## 7. Backtesting Simulation

### From accuracy to profitability

53% directional accuracy sounds useful. But does it translate to profit after -110 juice?
We simulate actual betting to find out.

### Half-Kelly criterion

Instead of betting a fixed amount, we size bets proportionally to our *edge*:

$$\text{edge} = p - 0.5238$$
$$\text{Kelly fraction} = \frac{\text{edge}}{0.9091}$$
$$\text{bet size} = \frac{\text{Kelly fraction}}{2} \times \text{bankroll}$$

We use *half*-Kelly (divide by 2) to reduce variance — full Kelly is theoretically optimal
but requires perfectly calibrated probabilities, which a logistic regression rarely provides.

We only bet when the model's confidence exceeds a threshold (default: 54%). Both sides are
considered: if the model strongly favors the away team, we bet the away side.

### Threshold sensitivity

A well-calibrated model should show *higher win rates* at higher confidence thresholds.
If we see the opposite, it means the model's probability estimates are poorly calibrated —
the raw probabilities don't reliably indicate certainty.


In [ ]:
JUICE        = 110
WIN_PAYOUT   = 100
PAYOUT_RATIO = WIN_PAYOUT / JUICE          # 0.9091
BREAKEVEN    = JUICE / (JUICE + WIN_PAYOUT) # 0.5238

def backtest(predictions, threshold=0.54, starting_bankroll=1000.0):
    bankroll = starting_bankroll
    season_rows = []

    for season, group in predictions.groupby('season'):
        bets, wins, wagered, profit = 0, 0, 0.0, 0.0

        for _, row in group.iterrows():
            prob = row['prob_home']
            bet_home = prob >= threshold
            bet_away = (1 - prob) >= threshold

            if not bet_home and not bet_away:
                continue

            p = prob if bet_home else (1 - prob)
            outcome_correct = (row['home_covered'] == 1) if bet_home else (row['home_covered'] == 0)

            edge       = p - BREAKEVEN
            kelly      = edge / PAYOUT_RATIO
            half_kelly = max(kelly / 2, 0)
            bet_size   = round(half_kelly * bankroll, 2)

            if bet_size <= 0:
                continue

            bets    += 1
            wagered += bet_size

            if outcome_correct:
                wins     += 1
                bankroll += bet_size * PAYOUT_RATIO
                profit   += bet_size * PAYOUT_RATIO
            else:
                bankroll -= bet_size
                profit   -= bet_size

        win_rate = wins / bets if bets > 0 else float('nan')
        roi      = profit / wagered * 100 if wagered > 0 else float('nan')
        season_rows.append({
            'season':   season,
            'bets':     bets,
            'win_rate': win_rate,
            'wagered':  round(wagered, 0),
            'profit':   round(profit, 0),
            'roi_%':    round(roi, 1),
            'bankroll': round(bankroll, 0),
        })

    return pd.DataFrame(season_rows), bankroll

# Threshold sensitivity
print("Threshold sensitivity (do higher thresholds = higher win rate?)")
print(f"  {'Threshold':>10}  {'Bets':>5}  {'Win%':>7}  {'ROI':>8}  {'Final $':>9}")
print(f"  {'─'*10}  {'─'*5}  {'─'*7}  {'─'*8}  {'─'*9}")
for thresh in [0.54, 0.56, 0.58, 0.60]:
    bt, final_bk = backtest(predictions, threshold=thresh)
    total_bets = bt['bets'].sum()
    total_w    = bt['wagered'].sum()
    total_p    = bt['profit'].sum()
    roi        = total_p / total_w * 100 if total_w > 0 else 0
    avg_win    = (bt['win_rate'] * bt['bets']).sum() / total_bets if total_bets > 0 else 0
    print(f"  {thresh:.0%}        {int(total_bets):>5}  {avg_win:>7.1%}  {roi:>+7.1f}%  ${final_bk:>8,.0f}")
print()
print(f"If win% FALLS as threshold rises → model probabilities are uncalibrated.")


In [ ]:
# Primary backtest at 54% threshold
bt_results, final_bankroll = backtest(predictions, threshold=0.54)

print(f"  {'Season':>6}  {'Bets':>4}  {'Win%':>6}  {'Wagered':>9}  {'Profit':>8}  {'ROI%':>6}  {'Bankroll':>10}")
print(f"  {'─'*6}  {'─'*4}  {'─'*6}  {'─'*9}  {'─'*8}  {'─'*6}  {'─'*10}")
for _, row in bt_results.iterrows():
    print(f"  {int(row['season']):>6}  {int(row['bets']):>4}  "
          f"{row['win_rate']:>6.1%}  ${row['wagered']:>8,.0f}  "
          f"{'+'if row['profit']>=0 else ''}{row['profit']:>7,.0f}  "
          f"{row['roi_%']:>+6.1f}%  ${row['bankroll']:>9,.0f}")
print(f"  {'─'*6}  {'─'*4}  {'─'*6}  {'─'*9}  {'─'*8}  {'─'*6}  {'─'*10}")
total_bets    = bt_results['bets'].sum()
total_wagered = bt_results['wagered'].sum()
total_profit  = bt_results['profit'].sum()
overall_roi   = total_profit / total_wagered * 100 if total_wagered > 0 else 0
print(f"  {'TOTAL':>6}  {int(total_bets):>4}  {'':>6}  "
      f"${total_wagered:>8,.0f}  {'+'if total_profit>=0 else ''}{total_profit:>7,.0f}  "
      f"{overall_roi:>+6.1f}%  ${final_bankroll:>9,.0f}")
print(f"\n  Starting bankroll: $1,000  →  Final: ${final_bankroll:,.0f}")
print(f"  Breakeven win rate: {BREAKEVEN:.1%}")


---
## 8. Conclusions

### What we found

| Finding | Result |
|---|---|
| Statistically significant features | 1 of 9 (`diff_def_epa`, p = 0.020) |
| Walk-forward accuracy (mean) | ~53.2% |
| Historical cover rate (baseline) | ~47.6% |
| Backtesting ROI (half-Kelly, 54% threshold) | −5.5% |
| Final bankroll ($1,000 start) | ~$490 |
| Win rate vs. 52.38% breakeven | Below breakeven |

### Interpretation

The model demonstrates that **publicly available NFL metrics contain almost no
information not already priced into the betting market**.

- Defensive EPA differential shows a weak but statistically significant correlation (p = 0.020)
  with home team covering. This is consistent with published academic research (Borghesi, 2007;
  Dare & McDonald, 1996) that finds very limited and inconsistent market inefficiencies in NFL spreads.

- The 53.2% directional accuracy *looks* useful but is **insufficient to overcome the -110 juice**.
  A bettor needs 52.38% win rate just to break even; 53.2% translates to roughly −$1.70 per \$110 risked.

- The threshold sensitivity test revealed **uncalibrated probabilities**: win rate *fell* as
  confidence threshold rose, indicating the model's raw probability scores don't reliably indicate
  certainty. This is a known limitation of logistic regression on noisy data.

- QB-adjusted EPA and opening line movement — two sophisticated features — showed no signal at all.
  The market is efficient with respect to both starter injury information and sharp money direction.

### What this means for the market efficiency hypothesis

> **The NFL spread betting market largely satisfies the weak form of market efficiency
> with respect to publicly available quantitative metrics.**

This doesn't mean the market is *perfectly* efficient — professional syndicates with
proprietary injury data, weather modeling, and real-time line shopping may find small edges.
But those edges are not accessible via freely available data.

### Skills demonstrated

- Data pipeline: multi-source merge of schedule, play-by-play, and historical odds data
- Feature engineering: rolling statistics with season-boundary resets and lookahead prevention
- Validation methodology: walk-forward cross-validation appropriate for time-series data
- Statistical analysis: correlation testing with p-values, Bonferroni correction awareness
- Kelly criterion: theoretically grounded bet sizing under uncertainty
- Negative result communication: honest documentation of null findings
